# Prepare stimuli_videos_roundx.csv for SAM3 annotation

Reads Datavyu annotation CSVs from `data/segmentation/` and produces
`data/stimuli_videos_roundx.csv` — the primary input to `annotate_video.py`.

Each row corresponds to one annotation prompt with its time window, text prompt,
and foreground/background pixel coordinates in video space.

| column | type | description |
|--------|------|-------------|
| `video_block` | str | clip name |
| `video_path` | str | absolute path to `_stripped.mp4` |
| `prompt_idx` | str | Datavyu prompt index (may contain letters, e.g. `"14a"`) |
| `prompt_name` | str | annotation label from Datavyu |
| `start_time_ms` / `end_time_ms` | int | clip time window in milliseconds |
| `start_time_s` / `end_time_s` | float | same, in seconds |
| `faces_and_hands` | bool | True if `face` or `hand` appears in the prompt name |
| `include_coords` | JSON | `[[x,y],…]` foreground points in video pixels |
| `exclude_coords` | JSON | `[[x,y],…]` background points in video pixels |

## Coordinate systems

| clip | coordinate origin | mapping |
|------|-------------------|---------|
| Sesame US 1 | full 1920×1080 screen | `vx = sx/1920*W`, `vy = sy/1080*H` |
| Pixar Birds, Sesame India 2, Slow People | video letterboxed in (46,53)–(1873,1079) on 1920×1080 | `vx = (sx−46)/1827*W`, `vy = (sy−53)/1026*H` |

In [ ]:
import json
import os
import sys
import cv2
import pandas as pd
from pathlib import Path

## Configuration

Auto-detects repo root by walking upward until `data/` is found.
Set `REPO_ROOT` manually below if auto-detection fails.

In [ ]:
_here = Path(os.getcwd())
REPO_ROOT = _here
while not (REPO_ROOT / 'data').is_dir() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

# REPO_ROOT = Path('/labs/vislearnlab/experiments/movie-watching')  # override if needed

CURRENT_ROUND = "round1" # or set to round2, round3 as needed
SEG_DIR     = REPO_ROOT / 'data' / 'segmentation' / f"{CURRENT_ROUND}" 
STIMULI_DIR = REPO_ROOT / 'stimuli' / 'main_blocks'
OUT_CSV     = REPO_ROOT / 'data' / 'segmentation' / f"stimuli_prompts_{CURRENT_ROUND}.csv"

print(f'Repo root   : {REPO_ROOT}')
print(f'Seg dir     : {SEG_DIR}')
print(f'Stimuli dir : {STIMULI_DIR}')
print(f'Output CSV  : {OUT_CSV}')

assert SEG_DIR.is_dir(), f'Segmentation dir not found: {SEG_DIR}'

## File mapping and coordinate systems

In [ ]:
# CSV filename → (video_block_label, coordinate_system)
# 'fullscreen': coordinates are in 1920×1080 screen space == video space
# 'letterbox' : video is shown in a sub-region of the 1920×1080 screen : issue on Remora Mac Mini
CSV_FILES = {
    'Pixar_Birds.csv':    ('Pixar Birds',    'letterbox'),
    'Sesame_India_2.csv': ('Sesame India 2', 'letterbox'),
    'Sesame_US_1.csv':    ('Sesame US 1',    'fullscreen'),
    'Slow_People.csv':    ('Slow People',    'letterbox'),
    'Slow_Hands.csv':     ('Slow Hands',      'fullscreen'),
    'Frank_Complex.csv':  ('Frank Complex', 'fullscreen'),
    'Sesame_India_1.csv':  ('Sesame India 1', 'fullscreen'),
    'Sesame_US_2.csv':     ('Sesame US 2', 'fullscreen'),
    'Frank_Play.csv': ('Frank Play', 'fullscreen')
}

# video_block label → file stem in stimuli/main_blocks/
BLOCK_TO_STEM = {
    'Pixar Birds':    'pixar_birds',
    'Sesame India 2': 'sesameindia_2',
    'Sesame US 1':    'sesameus_1',
    'Slow People':    'slow_people',
    'Slow Hands':     'slow_hands',
    'Frank Complex':  'frank_complex',
    'Sesame India 1': 'sesameindia_1',
    'Sesame US 2': 'sesameus_2',
    'Frank Play' : 'frank_play'
}

# Letterbox region: video is displayed within this rectangle on the 1920×1080 screen.
# Applies to Pixar_Birds, Sesame_India_2, Slow_People.
# top-left  = (46, 53)
# bottom-right = (1873, 1079)
# region size  = 1827 × 1026 px
LETTERBOX = dict(left=46, top=53, right=1873, bottom=1079)

print('CSV file → video block mapping:')
for fname, (block, csys) in CSV_FILES.items():
    print(f'  {fname:25s} → {block:18s} [{csys}]')

## Helper functions

In [ ]:
def parse_coord_val(s) -> int:
    """Parse a coordinate string; strips thousands-separator commas ('1,857' → 1857)."""
    return int(str(s).replace(',', '').strip())


def screen_to_video(
    sx: int, sy: int, coord_system: str, video_w: int, video_h: int
) -> tuple[int, int]:
    """
    Convert screen-space (sx, sy) to video-pixel (vx, vy).

    fullscreen: video fills the full 1920×1080 screen.
    letterbox:  video is displayed in LETTERBOX sub-region of the screen.
    """
    if coord_system == 'fullscreen':
        vx = round(sx / 1920 * video_w)
        vy = round(sy / 1080 * video_h)
    else:  # letterbox
        region_w = LETTERBOX['right']  - LETTERBOX['left']   # 1827
        region_h = LETTERBOX['bottom'] - LETTERBOX['top']    # 1026
        vx = round((sx - LETTERBOX['left']) / region_w * video_w)
        vy = round((sy - LETTERBOX['top'])  / region_h * video_h)
    # Clamp to valid pixel range
    vx = max(0, min(vx, video_w - 1))
    vy = max(0, min(vy, video_h - 1))
    return vx, vy


def get_video_path(block: str) -> Path | None:
    """Return path to the video file for a given video block label."""
    stem = BLOCK_TO_STEM.get(block)
    if stem is None:
        return None
    for suffix in ['_stripped.mp4', '.mp4']:
        p = STIMULI_DIR / f'{stem}{suffix}'
        if p.exists():
            return p
    return None


def get_video_dims(video_path: Path) -> tuple[int, int]:
    """Return (width, height) of the video in pixels."""
    cap = cv2.VideoCapture(str(video_path))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return w, h


print('Helper functions defined.')

## Load and parse Datavyu CSVs

Each CSV has interleaved `coords.*` and `prompts.*` columns on the same rows.
They are independent: a row may have coords data, prompts data, or both.
We split them into two separate DataFrames and join via `prompt_idx`.

In [ ]:
COORD_COLS = [
    'coords.ordinal', 'coords.onset', 'coords.offset',
    'coords.prompt_idx', 'coords.x', 'coords.y', 'coords.include',
]
PROMPT_COLS = [
    'prompts.ordinal', 'prompts.onset', 'prompts.offset',
    'prompts.prompt_idx', 'prompts.prompt_name',
]

all_prompts_list = []
all_coords_list  = []

for csv_fname, (video_block, coord_system) in CSV_FILES.items():
    csv_path = SEG_DIR / csv_fname
    assert csv_path.exists(), f'Missing: {csv_path}'

    # dtype=str: prevents pandas from misreading '1,857' or alphanumeric prompt_idx
    raw = pd.read_csv(csv_path, dtype=str)
    # The trailing comma in the Datavyu header creates an extra unnamed column — drop it
    raw = raw.loc[:, ~raw.columns.str.startswith('Unnamed')]

    # ── Prompts rows (where prompts.ordinal is present) ───────────────────────
    p_mask = raw['prompts.ordinal'].notna()
    p = raw.loc[p_mask, PROMPT_COLS].copy()
    p['video_block']  = video_block
    p['coord_system'] = coord_system
    all_prompts_list.append(p)

    # ── Coords rows (where coords.ordinal is present) ─────────────────────────
    c_mask = raw['coords.ordinal'].notna()
    c = raw.loc[c_mask, COORD_COLS].copy()
    c['video_block']  = video_block
    c['coord_system'] = coord_system
    all_coords_list.append(c)

    print(f'{csv_fname}: {p_mask.sum():3d} prompts, {c_mask.sum():3d} coords')

prompts_df = pd.concat(all_prompts_list, ignore_index=True)
coords_df  = pd.concat(all_coords_list,  ignore_index=True)
print(f'\nTotal: {len(prompts_df)} prompts, {len(coords_df)} coords')
prompts_df.head()

## Load video dimensions

Needed to convert screen-space coordinates into video-pixel coordinates.

In [ ]:
video_dims:      dict[str, tuple[int, int]] = {}
video_paths_map: dict[str, Path | None]     = {}

print('Video path resolution and dimensions:')
for block in BLOCK_TO_STEM:
    vp = get_video_path(block)
    video_paths_map[block] = vp
    if vp is None or not vp.exists():
        print(f'  WARNING: video not found for "{block}" — using 1920×1080 fallback')
        video_dims[block] = (1920, 1080)
    else:
        w, h = get_video_dims(vp)
        video_dims[block] = (w, h)
        print(f'  {block:20s}  {w:4d}×{h:4d}  ({vp.name})')

## Aggregate coords and build output rows

For each prompt row, collect all matching coords (via `prompt_idx`),
convert screen → video pixel space, and encode as JSON lists.

In [ ]:
# Normalize prompt_idx for joining (strip whitespace, keep as string)
prompts_df['_pid'] = prompts_df['prompts.prompt_idx'].str.strip()
coords_df['_pid']  = coords_df['coords.prompt_idx'].str.strip()

# Group coords by (video_block, prompt_idx) for fast lookup
coord_groups = coords_df.groupby(['video_block', '_pid'])

output_rows = []
for _, prow in prompts_df.iterrows():
    block     = prow['video_block']
    pid       = prow['_pid']
    csys      = prow['coord_system']
    vw, vh    = video_dims[block]
    vpath     = video_paths_map[block]
    name      = str(prow['prompts.prompt_name']).strip()
    onset_ms  = int(prow['prompts.onset'])
    offset_ms = int(prow['prompts.offset'])
    name_lower = name.lower()

    include_px: list[list[int]] = []
    exclude_px: list[list[int]] = []

    grp_key = (block, pid)
    if grp_key in coord_groups.groups:
        grp = coord_groups.get_group(grp_key)
        for _, crow in grp.iterrows():
            try:
                sx = parse_coord_val(crow['coords.x'])
                sy = parse_coord_val(crow['coords.y'])
                vx, vy = screen_to_video(sx, sy, csys, vw, vh)
                if str(crow['coords.include']).strip().lower() == 'yes':
                    include_px.append([vx, vy])
                else:
                    exclude_px.append([vx, vy])
            except (ValueError, TypeError) as e:
                print(f'  WARNING: skipping bad coord ({crow["coords.x"]}, {crow["coords.y"]}): {e}')

    output_rows.append(dict(
        video_block     = block,
        video_path      = str(vpath) if vpath else '',
        prompt_idx      = pid,
        prompt_name     = name,
        start_time_ms   = onset_ms,
        end_time_ms     = offset_ms,
        start_time_s    = round(onset_ms  / 1000, 3),
        end_time_s      = round(offset_ms / 1000, 3),
        faces_and_hands = ('face' in name_lower or 'hand' in name_lower),
        include_coords  = json.dumps(include_px),
        exclude_coords  = json.dumps(exclude_px),
    ))

out_df = pd.DataFrame(output_rows)
print(f'Built {len(out_df)} prompt rows')
out_df.head(10)

## Remap non-numeric `prompt_idx` values

Datavyu occasionally has alphanumeric `prompt_idx` values (e.g. `"14a"`), inserted
when an annotator adds an extra point between existing ordinals. Downstream, SAM3
requires an integer `obj_id` for the points-only fallback prompt (it lands in a
`torch.int64` tensor internally), so any non-numeric `prompt_idx` is remapped to a
new unique integer — starting one past the max numeric `prompt_idx` seen **across
all video blocks**, not just within the current one, so remapped ids can never
collide with a numeric id from another clip.

In [ ]:
pid_series = out_df['prompt_idx'].astype(str).str.strip()
is_numeric = pid_series.str.fullmatch(r'\d+')

# Max numeric prompt_idx across ALL video blocks combined, so remapped ids
# can't collide with a numeric id belonging to a different clip.
max_numeric_id = pid_series[is_numeric].astype(int).max()

next_id = max_numeric_id + 1
remap: dict[str, str] = {}
for pid in pid_series[~is_numeric].unique():
    remap[pid] = str(next_id)
    next_id += 1

if remap:
    print(f'Remapping {len(remap)} non-numeric prompt_idx value(s) to integers ≥ {max_numeric_id + 1}:')
    for old, new in remap.items():
        print(f'  {old!r} → {new}')
else:
    print('No non-numeric prompt_idx values found — nothing to remap.')

out_df['prompt_idx'] = pid_series.replace(remap)

## Summary statistics

In [ ]:
print('=== Summary ===\n')
print('Per-video breakdown:')
for block, grp in out_df.groupby('video_block'):
    fh          = grp['faces_and_hands'].sum()
    total       = len(grp)
    has_include = (grp['include_coords'] != '[]').sum()
    has_exclude = (grp['exclude_coords'] != '[]').sum()
    print(f'  {block:20s}  {total:3d} prompts  '
          f'{fh:3d} face/hand  '
          f'{has_include:3d} with include coords  '
          f'{has_exclude:3d} with exclude coords')

print(f'\nTotal prompts              : {len(out_df)}')
print(f'Face/hand prompts          : {out_df["faces_and_hands"].sum()}')
print(f'Prompts with include coords: {(out_df["include_coords"] != "[]").sum()}')
print(f'Prompts with exclude coords: {(out_df["exclude_coords"] != "[]").sum()}')

## Save `stimuli_prompts_roundx.csv`

In [ ]:
out_df.to_csv(OUT_CSV, index=False)
print(f'Saved {len(out_df)} rows → {OUT_CSV}')
print(f'Columns: {out_df.columns.tolist()}')